In [0]:
from datetime import datetime
_inicio = datetime.now()
_notebook_actual = "constelacion"

In [0]:
from pyspark.sql import functions as F

drop_oms = ["Dim3", "Dim3Type", "Comments", "DataSourceDim", "DataSourceDimType"]

oms = (spark.table("bronze.json_oms")
       .drop(*drop_oms)
       .withColumn("anio", F.col("TimeDim").cast("int"))
       .withColumn("valor", F.col("NumericValue").cast("double")))

oms.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("stage.oms_indicadores")

print("stage.oms_indicadores:", spark.table("stage.oms_indicadores").count(), "filas")

stage.oms_indicadores: 1708 filas


In [0]:
wb = spark.table("bronze.json_worldbank").drop("unit", "obs_status")

wb = (wb
    .withColumn("indicador_id",   F.regexp_extract("indicator", r"'id':\s*'([^']*)'", 1))
    .withColumn("indicador_desc", F.regexp_extract("indicator", r"'value':\s*'([^']*)'", 1))
    .withColumn("pais_iso3",      F.col("countryiso3code"))
    .withColumn("anio",           F.col("date").cast("int"))
    .withColumn("valor",          F.col("value").cast("double")))

wb.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("stage.worldbank_indicadores")

print("stage.worldbank_indicadores:", spark.table("stage.worldbank_indicadores").count(), "filas")

stage.worldbank_indicadores: 450 filas


In [0]:
from pyspark.sql.window import Window

d = spark.table("bronze.gdrive_docs").toDF("variable", "codigo", "etiqueta", "_archivo_origen", "_fuente")
d = d.filter(~((F.col("codigo") == "Código") & (F.col("etiqueta") == "Etiqueta")))

w = Window.orderBy(F.monotonically_increasing_id()).rowsBetween(Window.unboundedPreceding, 0)
d = d.withColumn("variable", F.last("variable", ignorenulls=True).over(w))

d.filter(F.col("codigo").isNotNull()) \
 .write.format("delta").mode("overwrite") \
 .option("overwriteSchema", "true").saveAsTable("stage.dim_diccionario")

print("stage.dim_diccionario:", spark.table("stage.dim_diccionario").count(), "filas")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


stage.dim_diccionario: 1836 filas


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE dw.fact_indicador_pais_anio AS
SELECT
    anio,
    SpatialDim         AS pais_iso3,
    'WHO_OMS'           AS fuente,
    IndicatorCode       AS indicador_codigo,
    NULL                AS indicador_nombre,
    valor
FROM stage.oms_indicadores

UNION ALL

SELECT
    anio,
    pais_iso3,
    'WORLDBANK'         AS fuente,
    indicador_id        AS indicador_codigo,
    indicador_desc      AS indicador_nombre,
    valor
FROM stage.worldbank_indicadores
""")

print("dw.fact_indicador_pais_anio:", spark.table("dw.fact_indicador_pais_anio").count(), "filas")
spark.sql("SELECT fuente, COUNT(*) FROM dw.fact_indicador_pais_anio GROUP BY fuente").show()

dw.fact_indicador_pais_anio: 2158 filas
+---------+--------+
|   fuente|COUNT(*)|
+---------+--------+
|  WHO_OMS|    1708|
|WORLDBANK|     450|
+---------+--------+



In [0]:
_fin = datetime.now()
_filas = spark.table("dw.fact_indicador_pais_anio").count()

spark.sql(f"""
INSERT INTO dw.etl_control_log (notebook, fecha_inicio, fecha_fin, estado, filas_salida, es_idempotente, nota)
VALUES (
    '{_notebook_actual}', '{_inicio}', '{_fin}', 'EXITOSO', {_filas}, true,
    'Fact constellation independiente, grano pais-anio, sin FK a fact_defunciones'
)
""")
print(f"Log registrado: {_notebook_actual} | {_filas:,} filas | {_fin - _inicio}")

Log registrado: constelacion | 2,158 filas | 0:00:15.064529
